<center>
    
## [mlcourse.ai](mlcourse.ai) – دورة مفتوحة للتعلم الآلي 
### <center> المؤلف: أرتيم كوزنتسوف، ODS Slack te
    
## <center> استكشاف محادثات TED



**خطة البحث**
    - وصف مجموعة البيانات والميزات
    - تحليل البيانات الاستكشافية
    - التحليل البصري للميزات
    - الأنماط والرؤى وخصائص البيانات
    - المعالجة المسبقة للبيانات
    - اختيار متري
    - هندسة الميزات والوصف
    - التحقق من الصحة، وضبط المعلمة الفائقة
    - التحقق ومنحنيات التعلم
    - التنبؤ بمجموعة الصمود
    - اختيار النموذج
    - الاستنتاجات



### الجزء الأول. وصف مجموعة البيانات والميزات



TED هو منظم المؤتمر، الذي يعقد الأحداث حيث يمكن للأشخاص من مختلف المناطق إجراء حديث عام عن الأفكار المهمة. زادت شعبية TED في السنوات الأخيرة بشكل ملحوظ بسبب منشورات المحادثات المسجلة بالفيديو والصوت.
تم جمع مجموعة البيانات بواسطة Rounak Banik وتخزينها في Kaggle https://www.kaggle.com/rounakbanik/ted-talks/. وليس من المؤكد كيف تم جمعها عن طريق حذف الويب أو بواسطة TED api (مغلق الآن). تحتوي البيانات على المحادثات قبل 21 سبتمبر 2017.
تتكون مجموعة البيانات من ملفين:
ted_main.csv - البيانات الوصفية حول المحادثات والمتحدثين- التعليقات- عدد تعليقات المستوى الأول على الحديث (عدد)
- الوصف - دعاية مغالى فيها لما يدور حوله الحديث (سلسلة)
- المدة - مدة الحديث بالثواني (عدد)
- حدث - حدث TED/TEDx الذي جرت فيه المحاضرة (سلسلة)
- film_date - الطابع الزمني لنظام Unix للتصوير (التاريخ بتنسيق زمني لنظام Unix)
- اللغات - عدد اللغات التي يتوفر بها الحديث (عدد)
- main_speaker - المتحدث الأول في المحاضرة (سلسلة)
- الاسم - الاسم الرسمي لمحادثة TED. يتضمن العنوان والمتحدث. (سلسلة)
- num_speaker - عدد المتحدثين في المحاضرة (عدد)
- تاريخ_النشر - الطابع الزمني لنظام يونكس لنشر المحاضرة على موقع TED.com (التاريخ بتنسيق زمني لنظام يونكس)
- التقييمات - قاموس مقيد للتقييمات المختلفة المعطاة للحديث (ملهم، رائع، مذهل، إلخ) (json)
- نقاشات ذات صلة - قائمة قواميس المحادثات الموصى بمشاهدتها بعد ذلك (json)
- مهنة المتحدث - مهنة المتحدث الرئيسي (سلسلة)
- العلامات - المواضيع المرتبطة بالحديث (قائمة)
- عنوان - عنوان الحديث (سلسلة)
- url - عنوان URL للحديث (سلسلة)
- المشاهدات - عدد المشاهدات على الحديث (عدد)
Transcripts.csv - نصوص الحديث
- النص - النص الإنجليزي الرسمي للمحاضرة. (سلسلة)
- url - عنوان URL للحديث (سلسلة)
الهدف من هذا المشروع هو البحث عن كيفية التنبؤ بعدد المشاهدات.


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import seaborn as sns
import scipy.stats

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV, learning_curve
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge

DATA_PATH = '../data/'

# Set up seeds
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

%matplotlib inline

In [ ]:
plt.rcParams['figure.figsize'] = 12., 9.


### الجزء الثاني. تحليل البيانات الاستكشافية


In [ ]:
# Load data
df_ted_main = pd.read_csv(DATA_PATH + 'ted_main.csv.zip')
df_ted_transcripts = pd.read_csv(DATA_PATH + 'transcripts.csv.zip')

In [ ]:
df_ted_main.info()

In [ ]:
df_ted_transcripts.info()


تحتوي مجموعات البيانات على عدد مختلف من السجلات، لذلك ربما يكون هناك عدد أقل من النصوص مقارنة بالمحادثات.



#### فحص التكرارات


In [ ]:
df_ted_main[df_ted_main.duplicated()]

In [ ]:
df_ted_transcripts[df_ted_transcripts.duplicated()]


حسنًا، لدينا بعض منها في df_ted_transcripts، فلنقم بإزالتها.


In [ ]:
df_ted_transcripts = df_ted_transcripts.drop_duplicates()


#### دمج مجموعات البيانات


In [ ]:
df_ted_main.shape, df_ted_transcripts.shape

In [ ]:
df_ted = pd.merge(df_ted_main, df_ted_transcripts, how='left', on='url')
df_ted.shape

In [ ]:
df_ted.columns

In [ ]:
df_ted.head()

In [ ]:
DATE_COLUMNS = 'film_date', 'published_date'
for column in DATE_COLUMNS:
    df_ted[column] = pd.to_datetime(df_ted[column], unit='s')

In [ ]:
df_ted.info()


#### القيم المفقودة


يبدو أن لدينا نصًا لجميع المحادثات تقريبًا ولكن لدينا أيضًا قيم مفقودة. كما أن بعض قيم talk_ocupation مفقودة.



#### إعادة فحص زمالة المدمنين المجهولين


In [ ]:
for column in df_ted.columns:
    na_count = df_ted[column].isna().sum()
    if na_count > 0:
        print('%s : %s' % (column, na_count))


#### إحصائيات الأرقام الشائعة


In [ ]:
df_ted.describe()

In [ ]:
df_ted.median()


### الوصف


In [ ]:
df_ted['description'].nunique(), len(df_ted['description'])

In [ ]:
df_ted['description'].str.len().describe()

In [ ]:
df_ted['duration'].values[:100]


كل حديث له وصف فريد.



###الحدث


In [ ]:
df_ted['event'].value_counts()


كان لدينا أنواع مختلفة من الأحداث هنا مثل حدث TED2014 الأكثر شعبية. يمكننا أن نرى أحداث TED و TEDx، وبعض الأحداث تختلف عنها. دعونا التحقيق أكثر من ذلك بقليل.


In [ ]:
sorted(df_ted['event'].unique())

In [ ]:
sorted(df_ted[df_ted['event'].str.startswith('TEDx')]['event'].unique())

In [ ]:
sorted(df_ted[df_ted['event'].str.startswith('TEDx') == False]['event'].unique())


يمكننا إضافة بعض الميزات إلى أنواع مختلفة من الأحداث.


In [ ]:
def get_event_type(event):
    '''
    Returns type of event
    '''
    if not 'TED' in event:
        return 'NOT_TED'
    elif event.startswith('TEDx'):
        return 'TEDx'
    elif event.startswith('TED@'):
        return 'TED@'
    elif re.fullmatch('TED\d{4}', event) is not None:
        return 'TED_YEAR'
    else:
        return event.split()[0]

In [ ]:
df_ted['event'].apply(get_event_type).value_counts()


تحتوي ويكيبيديا على بعض المعلومات الإضافية حول أنواع المؤتمرات المختلفة https://en.wikipedia.org/wiki/TED_(conference)


In [ ]:
df_ted.columns


####تاريخ_الفيلم


In [ ]:
df_ted['film_date'].describe()


يعود تاريخ تصوير بعض المحادثات إلى عام 1972. فلنحاول العثور على المزيد.


In [ ]:
df_ted[df_ted['film_date'] < '2000-01-01']


يمكن ملاحظة أن هناك ثلاث محادثات ليست من TED وتم تصويرها قبل عام 1992.



#### اللغات


In [ ]:
df_ted['languages'].describe(), df_ted['languages'].median()


ومن المثير للاهتمام أن بعض المحادثات لها عدد لغة يساوي الصفر. دعونا التحقيق قليلا.


In [ ]:
df_ted[df_ted['languages'] == 0]

In [ ]:
df_ted[df_ted['languages'] == 0]['url'].values[:10]


معظمها عروض فنية، ولكن ليس كلها. أيضًا بالنسبة لتلك السجلات لا يوجد نص.



#### المتحدث الرئيسي


In [ ]:
df_ted['main_speaker'].value_counts()

In [ ]:
df_ted['main_speaker'].value_counts().describe()


معظم الناس يتحدثون في فعاليات TED مرة واحدة فقط.


In [ ]:
df_ted['main_speaker'].str.len().describe()

In [ ]:
df_ted[df_ted['main_speaker'].str.len() > 20]


في حالة وجود حقل main_speaker طويل، يمكننا أن نشك في وجود أكثر من مكبر صوت واحد.



#### الاسم


In [ ]:
df_ted['name'].nunique()


كل حديث له اسم فريد.


In [ ]:
df_ted['name'].str.len().describe()


#### num_speaker


In [ ]:
df_ted['num_speaker'].describe()


يقدم معظم الناس محادثاتهم بمفردهم.



#### تاريخ النشر


In [ ]:
df_ted['published_date'].describe()

In [ ]:
df_ted[df_ted['event'].str.startswith('TED')]['film_date'].min()


تاريخ النشر الأول هو 27-06-2006، لكن أول محادثة تم تصويرها كانت بتاريخ 02-02-1984. لذلك قد يكون من المثير للاهتمام إلقاء نظرة على الفترة الزمنية بين التصوير والنشر.


In [ ]:
(df_ted['published_date'] - df_ted['film_date']).describe()

In [ ]:
(df_ted['published_date'] - df_ted['film_date']).median()

In [ ]:
df_ted[(df_ted['published_date'] - df_ted['film_date']).dt.total_seconds() < 0]

ومن المثير للاهتمام، يبدو أن لدينا بعض الأخطاء في البيانات. السجلات المذكورة أعلاه موجودة حيث يكون موقع Published_date أقدم من تاريخ_الفيلم، وهو ما يحدث.



#### التقييمات



إنه تصنيف من موقع TED. يطلب TED من الأشخاص وصف الفيديو (الحديث) في ثلاث كلمات. العدد يعني ببساطة عدد الأشخاص الذين اختاروا الفئة.
لن نستخدم الحقل لأنه مرتبط ارتباطًا وثيقًا بـ "طرق العرض" المتغيرة المستهدفة.
المزيد من مشاهدات الفيديو حازت على تقييم عدد أكبر من الأشخاص.


In [ ]:
df_ted['ratings'].values[0]

In [ ]:
df_ted['ratings'].values[1]

In [ ]:
df_ted['ratings'].values[2]

In [ ]:
df_ted['ratings'].values[3]


#### محادثات ذات صلة



لن نستخدم هذا المجال في البحث نظرًا لتعقيده في التحليل. 


In [ ]:
df_ted['related_talks'].values[0]


#### مهنة المتحدث


In [ ]:
df_ted['speaker_occupation'].value_counts()

In [ ]:
df_ted['speaker_occupation'].str.len().describe()

In [ ]:
df_ted[df_ted['speaker_occupation'].str.len() > 50]['speaker_occupation']


المهن الأكثر شعبية هي من الفنون والأعمال والصحافة والهندسة المعمارية وعلم النفس.
يصف بعض الأشخاص أنفسهم بالعديد من أنواع المهن المختلفة. يمكن أن يكون عدد المهن ميزة لاحقًا.



#### العلامات


In [ ]:
df_ted['tags'].values[:5]

In [ ]:
df_ted['tags'] = df_ted['tags'].apply(lambda x: eval(x))

In [ ]:
df_ted['tags'].values.reshape(-1,1)

In [ ]:
type(df_ted['tags'].values[0])

In [ ]:
# Some code to flatten list of tags
df_ted['tags'].apply(pd.Series).reset_index().melt(id_vars='index').value.dropna().value_counts()


#### العنوان


In [ ]:
df_ted['title'].nunique()


كل حديث له عنوانه الخاص.


In [ ]:
df_ted['title'].str.len().describe()

In [ ]:
df_ted['title'].values[:5]


يبدو أن العنوان + main_speaker = الاسم


In [ ]:
df_ted[['name', 'main_speaker', 'title']].head()


#### رابط


In [ ]:
df_ted['url'].nunique()

In [ ]:
df_ted['url'].values[:5]

In [ ]:
sum(df_ted['url'].str.endswith('\n'))


ينتهي كل عنوان url بـ "\n"، لذلك يمكن تنظيفه.


In [ ]:
df_ted['url'] = df_ted['url'].str.strip()

In [ ]:
df_ted['url'].apply(lambda s: s.split('/')[0]).value_counts()

In [ ]:
df_ted['url'].apply(lambda s: s.split('/')[2]).value_counts()

In [ ]:
df_ted['url'].apply(lambda s: s.split('/')[3]).value_counts()


جميع عناوين url هي 'https://www.ted.com/talks/name_of_talk' حتى نتمكن من حذف الحقل دون عواقب.



#### مشاهدات



"طرق العرض" هي المتغير المستهدف. نحتاج أيضًا إلى التحقق من التوزيع الطبيعي.


In [ ]:
df_ted['views'].describe()


لا تبدو موزعة بشكل طبيعي. دعونا نتحقق من خلال المؤامرات واختبارات الإحصائيات.


In [ ]:
df_ted['views'].hist(bins=100);

In [ ]:
scipy.stats.normaltest(df_ted['views'])

In [ ]:
scipy.stats.shapiro(df_ted['views'])

In [ ]:
sm.qqplot(df_ted['views'], line='s');

In [ ]:
scipy.stats.normaltest(np.log(df_ted['views']))

In [ ]:
scipy.stats.shapiro(np.log(df_ted['views']))

In [ ]:
sm.qqplot(np.log(df_ted['views']), line='s');

In [ ]:
np.log(df_ted['views']).hist(bins=100);

In [ ]:
alpha = 0.001
p = scipy.stats.shapiro(np.log(np.log(df_ted['views'])))[1]

if p < alpha:  # null hypothesis: x comes from a normal distribution
    print("The null hypothesis can be rejected")
else:
    print("The null hypothesis cannot be rejected")


لا يبدو أننا نحصل على التوزيع الطبيعي بعد تطبيق اللوغاريتم، لكنه يبدو أقرب إلى ذلك بكثير. لذلك سنفترض أن المتغير المستهدف له توزيع طبيعي.


In [ ]:
df_ted.columns

In [ ]:
df_ted['target'] = np.log(df_ted['views'])


####النصوص


In [ ]:
df_ted['transcript'].nunique(), len(df_ted['transcript']), sum(df_ted['transcript'].isna())


ليس كل حديث له نص وكل نسخة فريدة من نوعها.


In [ ]:
df_ted['transcript'].str.len().describe()

In [ ]:
df_ted[df_ted['transcript'].str.len() < 200]['transcript'].values


حسنًا، يبدو أن بعض النصوص مأخوذة من الموسيقى.


### الجزء 3. التحليل البصري للميزات


In [ ]:
df_ted.columns

In [ ]:
df_ted.drop('views', axis=1, inplace=True)

In [ ]:
df_ted.drop('related_talks', axis=1, inplace=True)

In [ ]:
df_ted.drop('comments', axis=1, inplace=True)

In [ ]:
# Make separate dataframe for data preparation for plotting
df_plot = df_ted.copy()
df_plot['film_date_unix'] = df_ted['film_date'].astype(int)
df_plot['published_date_unix'] = df_ted['published_date'].astype(int)

In [ ]:
%%time

sns.pairplot(df_plot, diag_kind="kde", markers="+",
    plot_kws=dict(s=50, edgecolor="b", linewidth=1),
    diag_kws=dict(shade=True));


ارتباط واضح بين عدد اللغات وعدد المشاهدات.


In [ ]:
df_ted.columns

In [ ]:
df_plot.corr(method='pearson')

In [ ]:
sns.heatmap(df_plot.corr(method='pearson').abs(), annot=True)

In [ ]:
df_plot.corr(method='spearman')

In [ ]:
sns.heatmap(df_plot.corr(method='spearman').abs(), annot=True)

In [ ]:
plt.plot(df_plot['published_date_unix'])


لذلك، يتم فرز البيانات حسب تاريخ النشر


In [ ]:
plt.plot(df_plot['target'])

In [ ]:
plt.plot(df_plot['published_date_unix'], df_plot['target'])

In [ ]:
sns.countplot(df_ted['event']);

In [ ]:
plt.plot(df_plot.groupby(by='event')['target'].mean().sort_values(ascending=False), 'o-');


يبدو أن المتغير المتوسط المستهدف يتم توزيعه بشكل طبيعي تقريبًا فيما يتعلق باسم الحدث.


In [ ]:
sns.countplot(df_ted['main_speaker']);

In [ ]:
plt.plot(df_plot.groupby(by='main_speaker')['target'].mean().sort_values(ascending=False), 'o-');


تنطبق طبيعة التوزيع أيضًا على اسم المتحدث.


In [ ]:
sns.countplot(df_ted['speaker_occupation']);

In [ ]:
plt.plot(df_plot.groupby(by='speaker_occupation')['target'].mean().sort_values(ascending=False), 'o-');


يبدو مكبر الصوت أيضًا موزعًا بشكل طبيعي.



### الجزء الرابع. الأنماط والرؤى وخصائص البيانات 


ومن التحليل السابق لدينا الملاحظات التالية:
- لا يتم توزيع متغير "العرض" بشكل طبيعي وفقًا للاختبارات. بالنسبة للنموذج الخطي، ينبغي أن يكون أكثر ملاءمة لاستخدام المتغير المستهدف الموزع بشكل طبيعي، لذلك نطبق اللوغاريتم عليه. إنه لا يجعل التوزيع طبيعيًا ولكنه يبدو الآن أقرب إليه.
- يرتبط المتغير المستهدف الجديد الخاص بنا ارتباطًا وثيقًا بعدد اللغات. ربما يرجع ذلك إلى حقيقة أن المحادثات الأكثر شعبية غالبًا ما تتم ترجمتها إلى المزيد من اللغات. لأننا نقوم بتحليل الارتباط، لا يمكننا أن نقول ذلك على وجه اليقين دون بيانات إضافية. ولكن قد يكون من المفيد حذف متغير اللغة.
- لدينا أيضًا علاقة قوية بين تاريخ النشر وتاريخ الفيلم، لذا نحتاج إلى استبعاد أحدهما للحصول على توقعات أكثر دقة.
- نرى بوضوح أنواعًا مختلفة من الأحداث، لذا قد يكون من المفيد إضافة ميزة إضافية مع معلومات نوع الحدث
- عنوان URL والاسم زائدان عن الحاجة فقط لأنه يحتوي على معلومات متاحة أيضًا من حقول أخرى
- يتم فرز البيانات حسب تاريخ النشر، حتى نتمكن من استخدام TimeSeriesSplit حتى لا ننشغل بتسريب البيانات. كما أنه جيد لأننا مهتمون بالتنبؤ بالميزات لذا لا نحتاج إلى فرز البيانات.
- لدينا بعض الأخطاء في البيانات، عندما يكون تاريخ النشر أصغر من تاريخ_الفيلم، ولكن نظرًا لأننا مهتمون أكثر بتاريخ النشر وارتباط تاريخ النشر وتاريخ_الفيلم ارتباطًا وثيقًا، فسوف نستبعد تاريخ_الفيلم



### الجزء الخامس. المعالجة المسبقة للبيانات


سنقوم بنوع مختلف من المعالجة المسبقة لأنواع مختلفة من الأعمدة:
- سيتم قياس الأعمدة الرقمية باستخدام StandardScaler
- سيتم تحويل أعمدة النص إلى أحرف صغيرة وبعد ذلك يتم توجيهها باستخدام TfIdfVectorizer
- سيتم تحليل المتغيرات الفئوية (على غرار ترميز التسمية) ثم تحويلها باستخدام OneHotEncoder
- سيتم ملء القيم الفارغة في حقل النص بسلسلة "na".
- سيتم تحويل التاريخ إلى وقت يونكس واستخدامه كرقم رقمي
- سيتم تحويل صفائف العلامات إلى سلسلة واستخدامها كعمود سلسلة


In [ ]:
df_ted.columns

In [ ]:
# Only leave features filtred by assumptions from previous part
X = df_ted[['description', 'duration', 'event', 'languages',
       'main_speaker', 'num_speaker', 'published_date',
       'speaker_occupation', 'tags', 'title', 'transcript']].copy()
y = df_ted['target'].copy()

NUMERIC_COLUMNS = ['duration', 'languages']
DATE_COLUMNS = ['published_date']

# We will convert 'tags' column to string
TEXT_COLUMNS = ['description', 'tags', 'title', 'transcript']

CATEGORICAL_COLUMNS = ['event',
       'main_speaker', 'speaker_occupation', 'num_speaker']

# We will convert published_date back to unix time and use it like numeric column
for c in DATE_COLUMNS:
    X[c] = X[c].astype(int)

# We will use data columns simply as numeric_column, so 
NUMERIC_COLUMNS += DATE_COLUMNS

# StandardScaler will convert fields to float64 with warning, so we will do it before
for c in NUMERIC_COLUMNS:
    X[c] = X[c].astype(float)

# Convert tags to string
X['tags'] = X['tags'].apply(lambda tags: ' '.join(tags))

X['transcript'] = X['transcript'].fillna('na')

# Convert all text columns to lower case
for c in TEXT_COLUMNS:
    X[c] = X[c].str.lower()

# Factorize categorical_columns (similar to LabelEncoding)
for c in CATEGORICAL_COLUMNS:
    X[c] = X[c].factorize()[0]

In [ ]:
X.head()

In [ ]:
preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(categories='auto', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
    ('scaler', StandardScaler(), NUMERIC_COLUMNS),
    ('tfidf_0', TfidfVectorizer(), TEXT_COLUMNS[0]),
    ('tfidf_1', TfidfVectorizer(), TEXT_COLUMNS[1]),
    ('tfidf_2', TfidfVectorizer(), TEXT_COLUMNS[2]),
    ('tfidf_3', TfidfVectorizer(), TEXT_COLUMNS[3]),
])

In [ ]:
# It's crucial not sort splits, because we want to predict future (so no future data should be in train set)
# We don't need seed here because shuffle is disabled
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.3, shuffle=False)

In [ ]:
X_train.shape, X_valid.shape, y_train.shape, y_valid.shape


### الجزء السادس. اختيار المتري



بالنسبة لمهمة الانحدار، يوجد مقياسان الأكثر شيوعًا - RMSE وMAE.
$
\بداية{محاذاة}
RMSE = \sqrt{\frac{1}{n}\sum_{j=1}^{n}{(\hat{y} - y_j)^2}}
\النهاية{محاذاة}
$
$
\بداية{محاذاة}
MAE = \frac{1}{n}\sum_{j=1}^{n}{\lvert\hat{y} - y_j\rvert}
\النهاية{محاذاة}
$
تضع RMSE وزنًا أكبر على الأخطاء الأكبر في التنبؤات.
يميل RMSE إلى الزيادة أكثر من MAE مع حجم عينة أكبر.
في حالتنا، لا ينبغي التهديد بالأخطاء الأكبر بطريقة خاصة.
يعد تفسير MAE أكثر سهولة، خاصة وأن لدينا تحويل سجل لمتغير الهدف الأولي، لذلك يمكن النظر إلى exp(MAE) على أنه مضاعف للقيمة الحقيقية للمتغير الأصلي.
لذلك سوف نذهب مع MAE.



### الجزء السابع. هندسة الميزات ووصفها 



دعونا نجرب ريدج من sklearn.


In [ ]:
%%time
model_ridge = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('ridge', Ridge(random_state=RANDOM_SEED))
    ]
)


cv = GridSearchCV(model_ridge, param_grid={}, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_


سوف نستخدمها كأساس للمستقبل.



دعونا نبني ميزات جديدة:
- لين النص (لأن الهدف يمكن أن يعتمد على مقدار الوقت الذي يتحدث فيه مقدم العرض)
- نوع الحدث، لأنه من الواضح أن بعض الأحداث أكثر شعبية (مثل أحداث TED مقابل أحداث TEDx الإقليمية)
- تاريخ النشر ساعة، شهر، يوم من الأسبوع


In [ ]:
# Only leave features filtred by assumptions from previous part
X = df_ted[['description', 'duration', 'event', 'languages',
       'main_speaker', 'num_speaker', 'published_date',
       'speaker_occupation', 'tags', 'title', 'transcript']].copy()
y = df_ted['target'].copy()

X['transcript'] = X['transcript'].fillna('na')
X['transcript_len'] = X['transcript'].str.len()
X['event_type'] = X['event'].apply(get_event_type)
X['published_hour'] = X['published_date'].dt.hour
X['published_month'] = X['published_date'].dt.month
X['published_dayofweek'] = X['published_date'].dt.dayofweek


NUMERIC_COLUMNS = ['duration', 'languages',
                   'transcript_len'
                  ]
DATE_COLUMNS = ['published_date']

# We will convert 'tags' column to string
TEXT_COLUMNS = ['description', 'tags', 'title', 'transcript']

CATEGORICAL_COLUMNS = ['event',
       'main_speaker', 'speaker_occupation', 'num_speaker', 
                       'event_type',
                       'published_hour',
                       'published_month',
                       'published_dayofweek'
                      ]

# We will convert published_date back to unix time and use it like numeric column
for c in DATE_COLUMNS:
    X[c] = X[c].astype(int)

# We will use data columns simply as numeric_column, so 
NUMERIC_COLUMNS += DATE_COLUMNS

# Convert tags to string
X['tags'] = X['tags'].apply(lambda tags: ' '.join(tags))

# StandardScaler will convert fields to float64 with warning, so we will do it before
for c in NUMERIC_COLUMNS:
    X[c] = X[c].astype(float)

# Convert all text columns to lower case
for c in TEXT_COLUMNS:
    X[c] = X[c].str.lower()

# Factorize categorical_columns (similar to LabelEncoding)
for c in CATEGORICAL_COLUMNS:
    X[c] = X[c].factorize()[0]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.3, shuffle=False)

سنقوم باختبار الميزات الجديدة واحدة تلو الأخرى، باستخدام خاصية ColumnTransformer - حيث سيتم إسقاط الأعمدة، غير المذكورة في قائمة المحولات.



#### دعونا نحاول استبعاد تاريخ النشر


In [ ]:
%%time
NUMERIC_COLUMNS = ['duration', 'languages']

CATEGORICAL_COLUMNS = ['event',
       'main_speaker', 'speaker_occupation', 'num_speaker']

preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(categories='auto', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
    ('scaler', StandardScaler(), NUMERIC_COLUMNS),
    ('tfidf_0', TfidfVectorizer(), TEXT_COLUMNS[0]),
    ('tfidf_1', TfidfVectorizer(), TEXT_COLUMNS[1]),
    ('tfidf_2', TfidfVectorizer(), TEXT_COLUMNS[2]),
    ('tfidf_3', TfidfVectorizer(), TEXT_COLUMNS[3]),
])

model_ridge = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('ridge', Ridge(random_state=RANDOM_SEED))
    ]
)


cv = GridSearchCV(model_ridge, param_grid={}, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_


لدينا بعض التحسن في النتيجة، دعونا نستمر.



####نسخة_لين


In [ ]:
%%time
NUMERIC_COLUMNS = ['duration', 'languages', 'transcript_len'
                  ]

CATEGORICAL_COLUMNS = ['event',
       'main_speaker', 'speaker_occupation', 'num_speaker']

preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(categories='auto', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
    ('scaler', StandardScaler(), NUMERIC_COLUMNS),
    ('tfidf_0', TfidfVectorizer(), TEXT_COLUMNS[0]),
    ('tfidf_1', TfidfVectorizer(), TEXT_COLUMNS[1]),
    ('tfidf_2', TfidfVectorizer(), TEXT_COLUMNS[2]),
    ('tfidf_3', TfidfVectorizer(), TEXT_COLUMNS[3]),
])

model_ridge = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('ridge', Ridge(random_state=RANDOM_SEED))
    ]
)


cv = GridSearchCV(model_ridge, param_grid={}, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_


كانت القيمة السابقة -0.4767736055960914، لذلك لدينا بعض التحسين البسيط.



#### نوع_الحدث


In [ ]:
%%time
NUMERIC_COLUMNS = ['duration', 'languages', 'transcript_len']

CATEGORICAL_COLUMNS = [
       'main_speaker', 'speaker_occupation', 'num_speaker', 'event_type']

preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(categories='auto', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
    ('scaler', StandardScaler(), NUMERIC_COLUMNS),
    ('tfidf_0', TfidfVectorizer(), TEXT_COLUMNS[0]),
    ('tfidf_1', TfidfVectorizer(), TEXT_COLUMNS[1]),
    ('tfidf_2', TfidfVectorizer(), TEXT_COLUMNS[2]),
    ('tfidf_3', TfidfVectorizer(), TEXT_COLUMNS[3]),
])

model_ridge = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('ridge', Ridge(random_state=RANDOM_SEED))
    ]
)


cv = GridSearchCV(model_ridge, param_grid={}, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_


هذه هي أفضل نتيجة كروسفال جديدة لدينا



#### ساعة النشر


In [ ]:
%%time
NUMERIC_COLUMNS = ['duration', 'languages', 'transcript_len'
                  ]

CATEGORICAL_COLUMNS = ['event_type',
       'main_speaker', 'speaker_occupation', 'num_speaker', 'published_hour']

preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(categories='auto', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
    ('scaler', StandardScaler(), NUMERIC_COLUMNS),
    ('tfidf_0', TfidfVectorizer(), TEXT_COLUMNS[0]),
    ('tfidf_1', TfidfVectorizer(), TEXT_COLUMNS[1]),
    ('tfidf_2', TfidfVectorizer(), TEXT_COLUMNS[2]),
    ('tfidf_3', TfidfVectorizer(), TEXT_COLUMNS[3]),
])

model_ridge = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('ridge', Ridge(random_state=RANDOM_SEED))
    ]
)


cv = GridSearchCV(model_ridge, param_grid={}, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_


عدم تحسين أفضل الدرجات



####نشر_الشهر


In [ ]:
%%time
NUMERIC_COLUMNS = ['duration', 'languages', 'transcript_len'
                  ]

CATEGORICAL_COLUMNS = ['event_type',
       'main_speaker', 'speaker_occupation', 'num_speaker', 'published_month']

preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(categories='auto', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
    ('scaler', StandardScaler(), NUMERIC_COLUMNS),
    ('tfidf_0', TfidfVectorizer(), TEXT_COLUMNS[0]),
    ('tfidf_1', TfidfVectorizer(), TEXT_COLUMNS[1]),
    ('tfidf_2', TfidfVectorizer(), TEXT_COLUMNS[2]),
    ('tfidf_3', TfidfVectorizer(), TEXT_COLUMNS[3]),
])

model_ridge = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('ridge', Ridge(random_state=RANDOM_SEED))
    ]
)


cv = GridSearchCV(model_ridge, param_grid={}, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_


عدم تحسين النتيجة



#### Published_dayofweek


In [ ]:
%%time
NUMERIC_COLUMNS = ['duration', 'languages', 'transcript_len'
                  ]

CATEGORICAL_COLUMNS = ['event_type',
       'main_speaker', 'speaker_occupation', 'num_speaker', 'published_dayofweek']

preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(categories='auto', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
    ('scaler', StandardScaler(), NUMERIC_COLUMNS),
    ('tfidf_0', TfidfVectorizer(), TEXT_COLUMNS[0]),
    ('tfidf_1', TfidfVectorizer(), TEXT_COLUMNS[1]),
    ('tfidf_2', TfidfVectorizer(), TEXT_COLUMNS[2]),
    ('tfidf_3', TfidfVectorizer(), TEXT_COLUMNS[3]),
])

model_ridge = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('ridge', Ridge(random_state=RANDOM_SEED))
    ]
)


cv = GridSearchCV(model_ridge, param_grid={}, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_


عدم تحسين النتيجة



#### استنتاج بشأن هندسة الميزات



لقد وجدنا للتو ميزتين مفيدتين جديدتين:
- نوع_الحدث بدلاً من الحدث
-نسخة_لين
يُظهر التحقق المتقاطع على Ridge تحسنًا في النتيجة مع كليهما.



### الجزء 8. التحقق من الصحة وضبط المعلمات الفائقة



سوف نستخدم الميزات التي وجدناها وقمنا باختيارها بالفعل.


In [ ]:
%%time

X = df_ted[['description', 'duration', 'event', 'languages',
       'main_speaker', 'num_speaker',
       'speaker_occupation', 'tags', 'title', 'transcript']].copy()
y = df_ted['target'].copy()

X['transcript'] = X['transcript'].fillna('na')
X['transcript_len'] = X['transcript'].str.len()
X['event_type'] = X['event'].apply(get_event_type)
X.drop('event', axis=1, inplace=True)

NUMERIC_COLUMNS = ['duration', 'languages', 'transcript_len']

CATEGORICAL_COLUMNS = ['main_speaker', 'speaker_occupation', 'num_speaker', 'event_type']


# We will convert 'tags' column to string
TEXT_COLUMNS = ['description', 'tags', 'title', 'transcript']

# Convert tags to string
X['tags'] = X['tags'].apply(lambda tags: ' '.join(tags))

# StandardScaler will convert fields to float64 with warning, so we will do it before
for c in NUMERIC_COLUMNS:
    X[c] = X[c].astype(float)

# Convert all text columns to lower case
for c in TEXT_COLUMNS:
    X[c] = X[c].str.lower()

# Factorize categorical_columns (similar to LabelEncoding)
for c in CATEGORICAL_COLUMNS:
    X[c] = X[c].factorize()[0]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.3, shuffle=False)

In [ ]:
X_train.shape, X_valid.shape, y_train.shape, y_valid.shape

In [ ]:
preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(categories='auto', handle_unknown='ignore'), CATEGORICAL_COLUMNS),
    ('scaler', StandardScaler(), NUMERIC_COLUMNS),
    ('tfidf_0', TfidfVectorizer(), TEXT_COLUMNS[0]),
    ('tfidf_1', TfidfVectorizer(), TEXT_COLUMNS[1]),
    ('tfidf_2', TfidfVectorizer(), TEXT_COLUMNS[2]),
    ('tfidf_3', TfidfVectorizer(), TEXT_COLUMNS[3]),
])


دعونا نضبط ألفا (تسوية l1) لريدج.


In [ ]:
model_ridge = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('ridge', Ridge(random_state=RANDOM_SEED))
    ]
)

params = {
    
    'ridge__alpha' : np.logspace(-2, 5, num=8)
}

cv = GridSearchCV(model_ridge, param_grid=params, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_

In [ ]:
cv.best_params_

In [ ]:
def plot_param_tuning(params, param_name, cv, x_scale_log=False):

    plt.plot(params[param_name], cv.cv_results_['mean_train_score'], 'o-', label='train')
    plt.plot(params[param_name], cv.cv_results_['mean_test_score'], 'o-', label='test')

    plt.fill_between(params[param_name],
                     cv.cv_results_['mean_train_score'] - cv.cv_results_['std_train_score'],
                     cv.cv_results_['mean_train_score'] + cv.cv_results_['std_train_score'],
                     alpha=0.2
                    )
    plt.fill_between(params[param_name],
                     cv.cv_results_['mean_test_score'] - cv.cv_results_['std_test_score'],
                     cv.cv_results_['mean_test_score'] + cv.cv_results_['std_test_score'],
                     alpha=0.2
                    )
    if x_scale_log:
        plt.xscale('log')

    plt.legend();

In [ ]:
plot_param_tuning(params, 'ridge__alpha', cv, x_scale_log=True)
plt.xlabel('alpha')
plt.ylabel('neg_mean_absolute_error')
plt.title('Ridge alpha tuning');


من الصعب تحديد قيمة ألفا جيدة، بسبب النطاق الواسع في الانحراف المعياري وأحجام العينات المختلفة بسبب TimeSeriesSplit. لكن يمكننا أن نعتبر alpha=10^2 تخمينًا جيدًا لأنه يوجد هنا فرق بسيط بين عينة التدريب وعينة الاختبار.


In [ ]:
model_lgb = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('lgb', LGBMRegressor(random_state=RANDOM_SEED))
    ]
)

params = {
}

cv = GridSearchCV(model_lgb, param_grid=params, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_

In [ ]:
%%time

model_lgb = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('lgb', LGBMRegressor(random_state=RANDOM_SEED))
    ]
)

params = {
    'lgb__max_depth': [2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]
}

cv = GridSearchCV(model_lgb, param_grid=params, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_

In [ ]:
cv.best_params_

In [ ]:
plot_param_tuning(params, 'lgb__max_depth', cv)
plt.xlabel('max_depth')
plt.ylabel('neg_mean_absolute_error')
plt.title('Lgb max_depth tuning');


بالتأكيد لا يمكننا أن نقول شيئًا عن أفضل عمق أقصى لانحدار lgbm. سوف نقوم بضبط n_estimators.


In [ ]:
model_lgb = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('lgb', LGBMRegressor(random_state=RANDOM_SEED))
    ]
)

params = {
    'lgb__n_estimators': [10,20,30,40,50, 60, 70, 80, 90, 100, 150, 200, 300, 400, 500, 600, 700]
}

cv = GridSearchCV(model_lgb, param_grid=params, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_, cv.best_params_

In [ ]:
plot_param_tuning(params, 'lgb__n_estimators', cv)
plt.xlabel('n_estimators')
plt.ylabel('neg_mean_absolute_error')
plt.title('Lgb n_estimators tuning');

In [ ]:
model_lgb = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('lgb', LGBMRegressor(random_state=RANDOM_SEED))
    ]
)

params = {
    'lgb__n_estimators': [30],
    'lgb__num_leaves':np.linspace(10,51, num=10, dtype=int)
}

cv = GridSearchCV(model_lgb, param_grid=params, scoring='neg_mean_absolute_error', cv=TimeSeriesSplit(n_splits=5),
                 return_train_score=True, verbose=3)
cv.fit(X_train, y_train)

In [ ]:
cv.best_score_, cv.best_params_

In [ ]:
plot_param_tuning(params, 'lgb__num_leaves', cv)
plt.xlabel('num_leaves')
plt.ylabel('neg_mean_absolute_error')
plt.title('Lgb num_leaves tuning');


يبدو أننا لم نحقق أي نجاح واضح في ضبط lgbm لذا يمكننا فقط استخدام 'lgb__n_estimators': 30 كمعلمة.



#### الخلاصة


معلماتنا نتيجة لضبط المعلمة الفائقة:
- ريدج - ألفا: 10 (لكن 100 تبدو أفضل بسبب المسافة الأصغر بين القطار والاختبار)
- انحدار LGBM - عدد المقدرين: 30



### الجزء التاسع. التحقق من الصحة ومنحنيات التعلم



#### ريدج


In [ ]:
X_train.shape, y_train.shape

In [ ]:
%%time

model_ridge = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('ridge', Ridge(random_state=RANDOM_SEED, alpha=100))
    ]
)


train_sizes, train_scores, test_scores = \
    learning_curve(model_ridge, X_train, y_train, 
                   cv=TimeSeriesSplit(n_splits=5), scoring='neg_mean_absolute_error', random_state=RANDOM_SEED)

In [ ]:
def plot_learning_curve(train_sizes, train_scores, test_scores):
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)

    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1,
                     color="r")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r",
             label="Training score")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="g",
             label="Cross-validation score")

    plt.legend(loc="best")

In [ ]:
plot_learning_curve(train_sizes, train_scores, test_scores)
plt.xlabel('train_sizes')
plt.ylabel('neg_mean_absolute_error')
plt.title('Learning curve Ridge');


#### تراجع LGBM


In [ ]:
%%time

model_lgb = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('lgb', LGBMRegressor(random_state=RANDOM_SEED, n_estimators=30))
    ]
)

train_sizes, train_scores, test_scores = \
    learning_curve(model_lgb, X_train, y_train, 
                   cv=TimeSeriesSplit(n_splits=5), scoring='neg_mean_absolute_error', random_state=RANDOM_SEED)

In [ ]:
plot_learning_curve(train_sizes, train_scores, test_scores)
plt.xlabel('train_sizes')
plt.ylabel('neg_mean_absolute_error')
plt.title('Learning curve Ridge');


يميل LGBRegressor نحو الإفراط في التجهيز، في حين تميل نتائج قطار Ridge والتحقق من الصحة إلى النظر بشكل أقرب إلى بعضها البعض.



### الجزء العاشر. التنبؤ بمجموعة الإيقاف



دعونا نتحقق من نماذجنا في مجموعة الانتظار. تم إنتاج مجموعة التعليق من جميع البيانات وتتكون من بيانات 30% الأخيرة مرتبة حسب الوقت.



#### ريدج


In [ ]:
model_ridge.fit(X_train, y_train)

In [ ]:
ridge_mae_valid = mean_absolute_error(y_valid, model_ridge.predict(X_valid))
ridge_mae_valid


#### تراجع LGBM


In [ ]:
model_lgb.fit(X_train, y_train)

In [ ]:
lgb_mae_valid = mean_absolute_error(y_valid, model_lgb.predict(X_valid))
lgb_mae_valid


### الجزء 11. اختيار النموذج



دعونا نعيد فحص cross_val_score للنماذج.


In [ ]:
%%time

ridge_cv_score = cross_val_score(model_ridge, X_train, y_train, scoring='neg_mean_absolute_error',
                                 cv=TimeSeriesSplit(n_splits=5))
lgb_cv_score = cross_val_score(model_lgb, X_train, y_train, scoring='neg_mean_absolute_error',
                                 cv=TimeSeriesSplit(n_splits=5))

In [ ]:
ridge_cv_score.mean(), lgb_cv_score.mean()

In [ ]:
pd.DataFrame(index=['Ridge', 'LGBRegressor'], data = [
    [ridge_mae_valid, -ridge_cv_score.mean()],
    [lgb_mae_valid, -lgb_cv_score.mean()],
    ], columns = ['valid', 'cv_score'])


لدينا بعض الفوضى في النتيجة. لا يوجد نموذج يبدو وكأنه الفائز النظيف. ولكن في مخططات التعلم، تبدو ريدج أكثر استدامة. لذا ربما ينبغي لنا أن نختار ريدج كنموذج أساسي لمزيد من البحث.



### الجزء 12. الاستنتاجات



لقد أجرينا بعض الأبحاث الأولية حول مجموعة بيانات TED Talks. لم يتم توزيع متغير "طرق العرض" بشكل طبيعي لذلك استخدمنا لوغاريتمًا له.
بعد ضبط المعلمة واختيار النموذج، تمكن كل من Ridge وLGBM regressor من الحصول على حوالي 0.48 MAE عند التحقق المتبادل. على الرغم من حقيقة أن مجموعة LGBM في حالة الانتظار = الخارج تتفوق على Ridge، إلا أن Ridge تبدو أفضل عند التحقق من الصحة.
يمكن أن يكون النموذج مفيدًا للبحث في التنبؤ بشعبية محادثات TED التي يتم قياسها من خلال عدد المشاهدات.
طرق تحسين النموذج وتطويره:
- تطبيع النص
- تحويل Tf-IDF ngramm_range للحقول النصية
- استخدام PCA قبل LGBM
- حاول الحصول على المزيد من البيانات (يجب أن تتوفر المزيد من البيانات الجديدة)
- ضبط النموذج بشكل أكثر دقة
- بحث حول كيفية أداء النموذج بدون متغير "اللغة".